# Планирование последовательностей интенций поверх FB — воспроизведение в Colab

Перед запуском: **Среда выполнения → Сменить среду выполнения → GPU (T4)**.

Понадобится директория чекпоинта с `params_*.pkl` и `flags.json` (внутри должны
быть `modules_forward_repr`, `modules_backward_repr`, `modules_actor`,
`modules_high_actor`). Положите её на Google Drive и укажите путь в ячейке
«Настройки».

## 1. Установка

In [ ]:
!git clone --recursive https://github.com/YOUR_USERNAME/fb-multi-intention-planning.git
%cd fb-multi-intention-planning
!pip install -q -r requirements-colab.txt

In [ ]:
import jax
print('устройства jax:', jax.devices())
assert jax.devices()[0].platform == 'gpu', 'GPU не подключён: Среда выполнения -> Сменить среду выполнения'

## 2. Данные

In [ ]:
!python scripts/download_datasets.py --datasets antmaze-medium-navigate-v0

## 3. Настройки

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ЗАМЕНИТЕ на путь к вашему чекпоинту.
CHECKPOINT = '/content/drive/MyDrive/checkpoints/antmaze-medium'
ENV = 'ogbench-antmaze-medium-navigate-v0'

import os
assert os.path.isdir(CHECKPOINT), f'нет директории {CHECKPOINT}'
print(sorted(os.listdir(CHECKPOINT)))

## 4. Проверки перед прогоном

Тесты логики планирования (чекпоинт не нужен) и калибровка масштабов среды.

In [ ]:
!python tests/test_planning.py
!python scripts/calibrate.py

## 5. E1: проверка гипотезы про горизонт

Запускается первым намеренно. Если прямая FB-оценка НЕ теряет контраст с
дальностью, значит гипотеза неверна, и метод надо переосмысливать здесь, а не
после всех прогонов.

In [ ]:
!python scripts/analysis_composability.py --checkpoint_dir "$CHECKPOINT" --env_name $ENV

## 6. E3: основной результат

In [ ]:
!python scripts/run_eval.py \
    --checkpoint_dir "$CHECKPOINT" --env_name $ENV \
    --methods baseline,graph,flat \
    --seeds 0,1,2,3,4 --num_episodes 20 --tag main

## 7. E2 и E5: карты ценности и качество графа

In [ ]:
!python scripts/analysis_value_maps.py --checkpoint_dir "$CHECKPOINT" --env_name $ENV --task_id 1
!python scripts/analysis_value_maps.py --checkpoint_dir "$CHECKPOINT" --env_name $ENV --task_id 3

## 8. E4: абляции

In [ ]:
!python scripts/run_ablations.py \
    --checkpoint_dir "$CHECKPOINT" --env_name $ENV \
    --seeds 0,1,2 --num_episodes 20 --include_baseline

## 9. Графики

In [ ]:
!python scripts/make_figures.py

import glob
from IPython.display import Image, display
for path in sorted(glob.glob('results/figures/*.png')):
    print(path)
    display(Image(path))

## 10. Сохранить результаты на Drive

Colab обнуляет диск после отключения — сырые csv лучше забрать сразу.

In [ ]:
!mkdir -p /content/drive/MyDrive/fbplan_results
!cp -r results/raw results/figures /content/drive/MyDrive/fbplan_results/
!ls -R /content/drive/MyDrive/fbplan_results | head -40